# Entrenamiento y comparación (issue #10)
Wrapper delgado: llama al mismo código del backend (`adapters.models`). No duplica preparación ni reescribe el CSV. Metodología del ticket: CV estratificada solo en train (5 pedidos), GridSearchCV chica, contraste en validación, test un solo uso.


In [ ]:
import sys
from pathlib import Path

CANDIDATOS = [Path.cwd() / "backend" / "src", Path.cwd().parent / "backend" / "src"]
SRC = next(c for c in CANDIDATOS if (c / "collection").is_dir())
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import collection
print("src:", SRC.exists(), "schema:", collection.SCHEMA_VERSION)


In [ ]:
ROOTS = [Path.cwd(), Path.cwd().parent]
DATASET = next(r / "dataset" / "salida" / "tramos.csv" for r in ROOTS if (r / "dataset" / "salida" / "tramos.csv").is_file())
print("dataset:", DATASET)


In [ ]:
from collection.adapters.datasets.ml_preparation import prepare_dataset

prepared = prepare_dataset(DATASET, seed=0)
print("sizes:", {k: v.metadata["size"] for k, v in prepared.partitions.items()})
print("dist:", prepared.metadata["class_distribution"])
assert sum(v.metadata["size"] for v in prepared.partitions.values()) == len(prepared.enriched) == 297
assert not hasattr(prepared.preprocessor, "transformers_")


In [ ]:
from collection.adapters.models.sklearn_evaluator import SklearnEvaluator
from collection.application.models.evaluation_protocol import EvaluationProtocol

evaluator = SklearnEvaluator()
comparison, rows = evaluator.evaluate_preparation(prepared, ["arbol", "logistica", "knn", "svc"], EvaluationProtocol())
for m in comparison.metrics:
    print(f"{m.model_id}: cv_f1={m.cv_f1_macro_mean:.4f} val_f1={m.validation_f1_macro:.4f} best={m.best_params}")
    print(f"  matriz_val={list(list(r) for r in m.validation_confusion_matrix)}")
print("pedidos/usados:", comparison.record["cv_folds_pedidos"], comparison.record["cv_folds_usados"])
print("elegido:", comparison.chosen_model_id)
assert comparison.record["cv_folds_pedidos"] == 5 and comparison.record["cv_folds_usados"] == 5
assert all(m.test_accuracy is None for m in comparison.metrics)
best = max(comparison.metrics, key=lambda m: (m.validation_f1_macro, m.validation_accuracy))
assert comparison.chosen_model_id == best.model_id


In [ ]:
final = evaluator.evaluate_final_on_test(comparison, rows, EvaluationProtocol())
chosen = next(m for m in final.metrics if m.model_id == final.chosen_model_id)
print(f"test un solo uso ({chosen.model_id}): acc={chosen.test_accuracy:.4f} f1={chosen.test_f1_macro:.4f}")
assert chosen.test_accuracy is not None and chosen.test_f1_macro is not None
assert sum(1 for m in final.metrics if m.test_accuracy is not None) == 1
print("NOTA: metricas = ajuste a la regla sintetica, no desempeno operativo.")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.tree import plot_tree
from IPython.display import display
from collection.adapters.models.model_specs import build_pipeline

ROOT = next(r for r in ROOTS if (r / "dataset" / "salida" / "tramos.csv").is_file())
FIGURES_DIR = ROOT / "docs" / "figs"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CLASS_ORDER = ("baja", "media", "alta")

# CV F1 macro (barras de error = std) frente a F1 macro de validación.
fig, ax = plt.subplots(figsize=(8, 5))
models = comparison.metrics
labels = [metric.model_id for metric in models]
positions = np.arange(len(models))
ax.bar(positions - 0.2, [m.cv_f1_macro_mean for m in models], width=0.4, yerr=[m.cv_f1_macro_std for m in models], capsize=4, label="CV F1 macro ± std")
ax.bar(positions + 0.2, [m.validation_f1_macro for m in models], width=0.4, label="Validación F1 macro")
ax.set(xticks=positions, xticklabels=labels, ylim=(0, 1.05), ylabel="F1 macro", title="CV (solo train) vs validación")
ax.legend(); fig.tight_layout()
path = FIGURES_DIR / "f1_comparacion.png"
fig.savefig(path, dpi=150); display(fig); plt.close(fig); print(path)

# Matrices de validación en orden baja/media/alta.
fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 4))
for ax, metric in zip(axes, models, strict=True):
    ConfusionMatrixDisplay(confusion_matrix=np.asarray(metric.validation_confusion_matrix), display_labels=CLASS_ORDER).plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")
    ax.set_title(metric.model_id)
fig.tight_layout()
path = FIGURES_DIR / "matrices_confusion_validacion.png"
fig.savefig(path, dpi=150); display(fig); plt.close(fig); print(path)

# Árbol ganador re-ajustado únicamente en train, mostrado hasta profundidad 3.
train_indices = prepared.partitions["train"].indices
tree = build_pipeline("arbol", prepared.preprocessor, seed=0)
tree.fit(prepared.X.iloc[list(train_indices)], prepared.y.iloc[list(train_indices)])
preprocessor_name = next(name for name in tree.named_steps if name != "decisiontreeclassifier")
fig, ax = plt.subplots(figsize=(18, 9))
plot_tree(tree.named_steps["decisiontreeclassifier"], feature_names=tree.named_steps[preprocessor_name].get_feature_names_out(), class_names=list(CLASS_ORDER), filled=True, rounded=True, max_depth=3, ax=ax)
ax.set_title("Árbol ganador (ajustado solo en train, profundidad máxima 3)")
fig.tight_layout()
path = FIGURES_DIR / "arbol_ganador.png"
fig.savefig(path, dpi=150); display(fig); plt.close(fig); print(path)
